In [1]:
import os
import zipfile

from urllib import request

import torch

from torch.utils.data import Dataset
from torchvision.models import resnet18, ResNet18_Weights
from torchvision import transforms

from PIL import Image

from typing import Callable

In [2]:
GPU = torch.device('cuda:0')

In [18]:
class ImageNetSketch(Dataset):
    URL = 'https://www.kaggle.com/api/v1/datasets/download/wanghaohan/imagenetsketch'
    ARCHIVE_NAME = 'ImageNet-Sketch.zip'
    EXTRACTED_FOLDER = 'imagenet-sketch/sketch'

    def __init__(self, root: str, transform: Callable | None = None, download: bool = False):
        self.root = os.path.expanduser(root)
        self.transform = transform
        self.dataset_dir = os.path.join(self.root, self.EXTRACTED_FOLDER)

        if download:
            self._download_and_extract()

        if not os.path.isdir(self.dataset_dir):
            raise RuntimeError(
                'Dataset not found. Set download=True to download it.'
            )

        self.classes = sorted(
            d.name for d in os.scandir(self.dataset_dir) if d.is_dir()
        )
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        self.samples = []
        for cls in self.classes:
            cls_dir = os.path.join(self.dataset_dir, cls)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append(
                        (os.path.join(cls_dir, fname), self.class_to_idx[cls])
                    )

    @staticmethod
    def _bytes_to_human(bytes_value):
        for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
            if bytes_value < 1024 or unit == 'TB':
                return f'{bytes_value:.2f} {unit}'
            bytes_value /= 1024

    @staticmethod
    def _reporthook(count, block_size, total_size):
        downloaded = count * block_size
        downloaded_str = ImageNetSketch._bytes_to_human(downloaded)

        if total_size <= 0:
            print(f'\rDownloaded: {downloaded_str}', end='', flush=True)
            return

        total_str = ImageNetSketch._bytes_to_human(total_size)
        percent = min(100, (downloaded * 100) / total_size)

        print(f'\rProgress: {percent:.1f}% ({downloaded_str} / {total_str})',
              end='', flush=True)

        if percent >= 100:
            print(f'\nDownload complete! Total: {total_str}')

    def _download_and_extract(self):
        os.makedirs(self.root, exist_ok=True)
        archive_path = os.path.join(self.root, self.ARCHIVE_NAME)

        if not os.path.exists(self.dataset_dir):
            if not os.path.exists(archive_path):
                print('Downloading ImageNet-Sketch...')
                request.urlretrieve(self.URL, archive_path, self._reporthook)

            print('Extracting ImageNet-Sketch...')
            with zipfile.ZipFile(archive_path, 'r') as zipf:
                zipf.extractall(self.root)
            print('Extracted!')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[Image.Image, str]:
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

In [4]:
def get_resnet_class(probs: torch.tensor) -> list[str]:
    categories = ResNet18_Weights.IMAGENET1K_V1.meta['categories']
    
    if probs.dim == 1:
        probs = probs.unsqueeze(0)
    
    return [categories[idx] for idx in probs.argmax(dim=1)]

In [5]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [19]:
dataset = ImageNetSketch('./data', transform=preprocess, download=True)

In [ ]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).to(GPU)
model.eval()

In [ ]:
img = Image.open('cat.jpg').convert('RGB')

In [ ]:
input_tensor = preprocess(img)
input_tensor = input_tensor.unsqueeze(0)
input_tensor = input_tensor.to(GPU)

In [ ]:
with torch.no_grad():
    output = model(input_tensor)

In [ ]:
output.shape, get_resnet_class(output)

btw, it's a cat...

In [ ]:
ResNet18_Weights.IMAGENET1K_V1.meta['_metrics']